## Импорт необходимых библиотек

In [ ]:
!pip install -r "requirements.txt"

## Загружаем данные

Загружаем граф из файла, записывая его вершины и ребра, а также создавая словарь для смежных вершин.

In [16]:
def load_graph(file, directed=False):
    edges = []
    nodes = set()
    adjacency = {}

    with open(file, 'r') as file:
        for line in file:
            if line.startswith('#') or not line.strip():
                continue
            u, v = map(int, line.strip().split())

            edges.append((u, v))
            nodes.update([u, v])

            if u not in adjacency:
                adjacency[u] = set()
            adjacency[u].add(v)

            if not directed:
                if v not in adjacency:
                    adjacency[v] = set()
                adjacency[v].add(u)

    return edges, nodes, adjacency

## Вспомогательные функции

In [17]:
def dfs(adjacency, node, visited, component=None, post_order=None):
    stack = [(node, False)]
    while stack:
        curr_node, children_visited = stack.pop()
        if curr_node not in visited:
            if children_visited:
                if post_order is not None:
                    post_order.append(curr_node)
            else:
                visited.add(curr_node)
                if component is not None:
                    component.append(curr_node)
                stack.append((curr_node, True))
                for neighbor in adjacency.get(curr_node, []):
                    if neighbor not in visited:
                        stack.append((neighbor, False))

In [18]:
def reverse_graph(edges):
    reversed_graph = {}
    for u, v in edges:
        if v not in reversed_graph:
            reversed_graph[v] = set()
        reversed_graph[v].add(u)
    return reversed_graph

In [19]:
def count_links(adjacency: dict, node: int):
    neighbors = adjacency.get(node, set())
    links = 0
    neighbor_list = list(neighbors)
    n = len(neighbor_list)
    for i in range(n):
        u = neighbor_list[i]
        for j in range(i + 1, n):
            v = neighbor_list[j]
            if u in adjacency and v in adjacency[u]:
                links += 1
    return links

In [20]:
def count_triples(adjacency: dict):
    triples = 0.0
    for neighbors in adjacency.values():
        k = len(neighbors)
        triples += k * (k - 1) / 2
    return triples

Считаем общее число треугольников в графе (total_triangles) и средний кластерный коэффициент (```average_clustering```):

In [21]:
def compute_local_metrics(adjacency: dict):
    clustering_coeffs = []
    triangles_counted = 0

    for node, neighbors in adjacency.items():
        k = len(neighbors)    
        if k < 2:
            clustering_coeffs.append(0.0)
            continue

        links = count_links(adjacency, node)
        Ci = (2 * links) / (k * (k - 1))
        clustering_coeffs.append(Ci)

        triangles_counted += links

    total_triangles = triangles_counted // 3
    average_clustering = (
        sum(clustering_coeffs) / len(clustering_coeffs)
        if clustering_coeffs else 0.0
    )
    return total_triangles, average_clustering

## Анализ структуры сети

### Пункт 1

In [22]:
def graph_analysis_1(file, directed=False):
    edges, nodes, adjacency = load_graph(file, directed)
    
    num_nodes = len(nodes)
    num_edges = len(edges)
    
    max_edges = num_nodes * (num_nodes - 1)
    if not directed:
        max_edges //= 2
        
    density = num_edges / max_edges if max_edges else 0

    visited = set()
    weak_components = []
    for node in nodes:
        if node not in visited:
            component = []
            dfs(adjacency, node, visited, component)
            weak_components.append(component)

    num_weak_components = len(weak_components)
    share_max_weak_component = max(len(c) for c in weak_components) / num_nodes if num_nodes else 0

    print(f"Число вершин: {num_nodes}")
    print(f"Число ребер: {num_edges}")
    print(f"Плотность: {density:.5f}")
    print("--" * 20)
    print(f"Число компонент слабой связности: {num_weak_components}")
    if num_nodes <= 15:
            print(weak_components)
    print(f"Доля вершин в максимальной по мощности компоненте слабой связности: {share_max_weak_component:.5f}")
    
    if directed:
        reversed_graph = reverse_graph(edges)
        visited = set()
        post_order = []
        for node in nodes:
            if node not in visited:
                dfs(reversed_graph, node, visited, post_order)
                
        visited = set()
        strong_components = []
        for node in reversed(post_order):
            if node not in visited:
                component = []
                dfs(adjacency, node, visited, component)
                strong_components.append(component)
        
        num_strong_components = len(strong_components)
        share_max_strong_component = max(len(c) for c in strong_components) / num_nodes if num_nodes else 0
        
        print("--" * 20)  
        print(f"Число компонент сильной связности: {num_strong_components}")
        if num_nodes <= 15:
            print(strong_components)
        print(f"Доля вершин в максимальной компоненте сильной связности: {share_max_strong_component:.5f}")

In [23]:
graph_analysis_1("test_graph.txt", directed=True)

Число вершин: 14
Число ребер: 16
Плотность: 0.08791
----------------------------------------
Число компонент слабой связности: 3
[[1, 2, 3, 5, 4], [6, 7, 10, 12, 13, 8, 9, 11], [14]]
Доля вершин в максимальной по мощности компоненте слабой связности: 0.57143
----------------------------------------
Число компонент сильной связности: 7
[[14], [13], [12], [11], [10], [7, 8, 9, 6], [4, 2, 3, 5, 1]]
Доля вершин в максимальной компоненте сильной связности: 0.35714


### Пункт 3

In [24]:
def graph_analysis_3(file):
    _, _, adjacency = load_graph(file)
    total_triangles, avg_clustering = compute_local_metrics(adjacency)
    triples = count_triples(adjacency)
    global_clustering = 0.0
    if triples > 0:
        global_clustering = 3 * total_triangles / triples


    print(f"Число треугольников: {total_triangles}")
    print(f"Средний кластерный коэффициент: {avg_clustering:.5f}")
    print(f"Глобальный кластерный коэффициент: {global_clustering:.5f}")

In [25]:
graph_analysis_3("test_graph.txt")

Число треугольников: 1
Средний кластерный коэффициент: 0.11905
Глобальный кластерный коэффициент: 0.13636


### Пункт 4

Перейдем к задаче вычислению среднего кластерного коэффициента сети для наибольшей компоненты слабой связности.

Для этого: 
- найдем компоненты слабой связаности и выберем наибольшую из них
- перезапишем для этой компоненты словарь со смежными вершинами (в коде дополнительно создадим множество ``` largest_component_set ```, чтобы ускорить обращение к вершинам: ```v in largest_component_set```)
- передадим его (как в **3 пункте**) в функцию для вычислению среднего кластерного коэффициента.

In [26]:
def graph_analysis_4(file):
    _, nodes, adjacency = load_graph(file)
    
    visited = set()
    weak_components = []
    for node in nodes:
        if node not in visited:
            component = []
            dfs(adjacency, node, visited, component)
            weak_components.append(component)

    if not weak_components:
        print("Граф пуст или без ребер")
        return

    largest_component = max(weak_components, key=len)
    largest_component_set = set(largest_component)
    
    tmp_adjacency = {}
    
    for u in largest_component:
        neighbors = adjacency.get(u, set())
        tmp_set = set()
        for v in neighbors:
            if v in largest_component_set:
                tmp_set.add(v)
        tmp_adjacency[u] = tmp_set


    _, avg_clustering = compute_local_metrics(tmp_adjacency)

    print(f"Средний кластерный коэффициент наибольшей компоненты слабой связности: {avg_clustering:.5f}")

In [27]:
graph_analysis_4("test_graph.txt")

Средний кластерный коэффициент наибольшей компоненты слабой связности: 0.00000
